# Parsing a PDF into a data dictionary

The challenge here is deconstruct the code.  That is, to understand the code by reading through and running the code, then annotating the various sections and/or rewriting the code in your own way.  Ultimately, you should be able to start with a blank notebook and recreate the process of parsing the PDF into a data dictionary.

This challenge uses almost everything that we've covered so far in class ( i.e. strings, lists, dictionaries, looping, slicing, regular expressions, conditionals, HTTP requests, data frames ), plus a couple of additional items, e.g. the `BytesIO` module and the `:=` operator.

Please do ask in the Slack channel if you have any questions.


## Setup

In [1]:
%%capture output
!pip install --break-system-packages pdfplumber


In [2]:
import requests
import pdfplumber
import re
import pandas as pd
from io import BytesIO


## Read in the PDF

In [3]:
pdf_url = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Housing%20-%20Data%20Documentation.pdf"
pdf_url


'https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Housing%20-%20Data%20Documentation.pdf'

In [4]:
response = requests.get(pdf_url)
# response.raise_for_status()  # Check that the request was successful
response

<Response [200]>

In [5]:
pdf_file = BytesIO(response.content)

pdf_pages = [
  page
    for page in pdfplumber.open(pdf_file).pages
]

len(pdf_pages)


12

In [6]:
pdf_pages[0]


<Page:1>

In [8]:
print(pdf_pages[0].extract_text())


SIZE: 2930 observations, 82 variables
DESCRIPTIVE ABSTRACT: Data set contains information from the
Assessor's Office used in computing assessed values for individual
residential properties sold from 2006 to 2010.
VARIABLE DESCRIPTIONS:
Tab characters are used to separate variables in the data file. The
data has 82 columns which include 23 nominal, 23 ordinal, 14 discrete,
and 20 continuous variables (and 2 additional observation
identifiers).
Order (Discrete): Observation number
PID (Nominal): Parcel identification number
MS SubClass (Nominal): Identifies the type of dwelling involved in the
sale.
020 1-STORY 1946 & NEWER ALL STYLES
030 1-STORY 1945 & OLDER
040 1-STORY W/FINISHED ATTIC ALL AGES
045 1-1/2 STORY - UNFINISHED ALL AGES
050 1-1/2 STORY FINISHED ALL AGES
060 2-STORY 1946 & NEWER
070 2-STORY 1945 & OLDER
075 2-1/2 STORY ALL AGES
080 SPLIT OR MULTI-LEVEL
085 SPLIT FOYER
090 DUPLEX - ALL STYLES AND AGES
120 1-STORY PUD (Planned Unit Development) - 1946 & NEWER
150 1-1/2 STORY P

In [9]:
pdf_pages[0].extract_text().split("\n")


['SIZE: 2930 observations, 82 variables',
 'DESCRIPTIVE ABSTRACT: Data set contains information from the',
 "Assessor's Office used in computing assessed values for individual",
 'residential properties sold from 2006 to 2010.',
 'VARIABLE DESCRIPTIONS:',
 'Tab characters are used to separate variables in the data file. The',
 'data has 82 columns which include 23 nominal, 23 ordinal, 14 discrete,',
 'and 20 continuous variables (and 2 additional observation',
 'identifiers).',
 'Order (Discrete): Observation number',
 'PID (Nominal): Parcel identification number',
 'MS SubClass (Nominal): Identifies the type of dwelling involved in the',
 'sale.',
 '020 1-STORY 1946 & NEWER ALL STYLES',
 '030 1-STORY 1945 & OLDER',
 '040 1-STORY W/FINISHED ATTIC ALL AGES',
 '045 1-1/2 STORY - UNFINISHED ALL AGES',
 '050 1-1/2 STORY FINISHED ALL AGES',
 '060 2-STORY 1946 & NEWER',
 '070 2-STORY 1945 & OLDER',
 '075 2-1/2 STORY ALL AGES',
 '080 SPLIT OR MULTI-LEVEL',
 '085 SPLIT FOYER',
 '090 DUPLEX - A

## Extract the text lines from the PDF

In [10]:
pdf_lines = [
  line
    for page in pdf_pages
      for line in page.extract_text().split("\n")
]

len(pdf_lines)


436

In [11]:
pdf_lines[:20]


['SIZE: 2930 observations, 82 variables',
 'DESCRIPTIVE ABSTRACT: Data set contains information from the',
 "Assessor's Office used in computing assessed values for individual",
 'residential properties sold from 2006 to 2010.',
 'VARIABLE DESCRIPTIONS:',
 'Tab characters are used to separate variables in the data file. The',
 'data has 82 columns which include 23 nominal, 23 ordinal, 14 discrete,',
 'and 20 continuous variables (and 2 additional observation',
 'identifiers).',
 'Order (Discrete): Observation number',
 'PID (Nominal): Parcel identification number',
 'MS SubClass (Nominal): Identifies the type of dwelling involved in the',
 'sale.',
 '020 1-STORY 1946 & NEWER ALL STYLES',
 '030 1-STORY 1945 & OLDER',
 '040 1-STORY W/FINISHED ATTIC ALL AGES',
 '045 1-1/2 STORY - UNFINISHED ALL AGES',
 '050 1-1/2 STORY FINISHED ALL AGES',
 '060 2-STORY 1946 & NEWER',
 '070 2-STORY 1945 & OLDER']

## Filter for lines that contain the field names

In [12]:
field_lines = [
  line
    for line in pdf_lines
      if re.search(r'\(.*\)\s*:', line)
]

len(field_lines)


82

In [13]:
field_lines[:10]


['Order (Discrete): Observation number',
 'PID (Nominal): Parcel identification number',
 'MS SubClass (Nominal): Identifies the type of dwelling involved in the',
 'MS Zoning (Nominal): Identifies the general zoning classification of',
 'Lot Frontage (Continuous): Linear feet of street connected to property',
 'Lot Area (Continuous): Lot size in square feet',
 'Street (Nominal): Type of road access to property',
 'Alley (Nominal): Type of alley access to property',
 'Lot Shape (Ordinal): General shape of property',
 'Land Contour (Nominal): Flatness of the property']

## Parse out the field name, type, and description

In [14]:
pattern = re.compile(r"^(.+)\s+\((\w+)\)\s*:\s+([^(]*)")

fields = [
  {
    "name": match.group(1),
    "type": match.group(2),
    "description": match.group(3),
  }
  for entry in field_lines
    if (match := pattern.search(entry))
]

len(fields)


82

In [15]:
fields[:5]


[{'name': 'Order', 'type': 'Discrete', 'description': 'Observation number'},
 {'name': 'PID',
  'type': 'Nominal',
  'description': 'Parcel identification number'},
 {'name': 'MS SubClass',
  'type': 'Nominal',
  'description': 'Identifies the type of dwelling involved in the'},
 {'name': 'MS Zoning',
  'type': 'Nominal',
  'description': 'Identifies the general zoning classification of'},
 {'name': 'Lot Frontage',
  'type': 'Continuous',
  'description': 'Linear feet of street connected to property'}]

## Convert to a data frame

In [16]:
data_dictionary = pd.DataFrame( fields )
data_dictionary.head()


,name,type,description
0,Order,Discrete,Observation number
1,PID,Nominal,Parcel identification number
2,MS SubClass,Nominal,Identifies the type of dwelling involved in the
3,MS Zoning,Nominal,Identifies the general zoning classification of
4,Lot Frontage,Continuous,Linear feet of street connected to property


## Query the data frame

In [17]:
data_dictionary["type"].value_counts()


,count
type,
Nominal,24
Ordinal,23
Continuous,20
Discrete,15


In [18]:
len(set(data_dictionary["name"]))


82

## All-In-One

This does ( mostly ) everything the above code cells do in a single cell and a single list comprehension.

In [19]:
pdf_url = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Housing%20-%20Data%20Documentation.pdf"
response = requests.get(pdf_url)
pdf_file = BytesIO(response.content)

pattern = re.compile(r"^(.+)\s+\((\w+)\)\s*:\s+([^(]*)")

data_dictionary = pd.DataFrame(
  {
    "name": match.group(1),
    "type": match.group(2),
    "description": match.group(3),
  }
    for page in pdfplumber.open(pdf_file).pages
      for line in page.extract_text().split("\n")
        if (match := pattern.search(line) )
)

data_dictionary.shape


(82, 3)

In [20]:
data_dictionary[:5]

,name,type,description
0,Order,Discrete,Observation number
1,PID,Nominal,Parcel identification number
2,MS SubClass,Nominal,Identifies the type of dwelling involved in the
3,MS Zoning,Nominal,Identifies the general zoning classification of
4,Lot Frontage,Continuous,Linear feet of street connected to property


## Join with data

In [21]:
data_url = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv"
data_url


'https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv'

In [22]:
data_df = pd.read_csv( data_url )
data_df


,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,905101070,20,RL,62.0,14299,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,7,2007,WD,Normal,115400
1,905101330,90,RL,72.0,10791,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,Shed,500,10,2006,WD,Normal,90000
2,903454090,50,RM,50.0,9000,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,12,2007,WD,Normal,141000
3,533244030,60,FV,68.0,7379,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,254000
4,909252020,70,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2009,WD,Normal,155000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,903231070,50,RM,52.0,6240,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,Shed,400,9,2006,WD,Normal,114500
2633,906201021,80,RL,74.0,10778,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2009,WD,Normal,162000
2634,533253070,120,RL,61.0,3782,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2009,WD,Normal,211500
2635,527376100,20,RL,78.0,10140,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,8,2009,WD,Normal,165000


In [23]:
cols_df = pd.DataFrame( data_df.columns )
cols_df


,0
0,PID
1,MS SubClass
2,MS Zoning
3,Lot Frontage
4,Lot Area
...,...
76,Mo Sold
77,Yr Sold
78,Sale Type
79,Sale Condition


In [24]:
pd.set_option('display.max_rows', None)
joined = (
    cols_df
        .set_index(0)
        .join( data_dictionary.set_index( "name" ), how="outer" )
        .reset_index( names=["col"])
)
joined


,col,type,description
0,1st Flr SF,Continuous,First Floor square feet
1,2nd Flr SF,Continuous,Second floor square feet
2,3-Ssn Porch,Continuous,Three season porch area in square feet
3,3Ssn Porch,NaN,NaN
4,Alley,Nominal,Type of alley access to property
5,Bedroom,Discrete,Bedrooms above grade
6,Bedroom AbvGr,NaN,NaN
7,Bldg Type,Nominal,Type of dwelling
8,Bsmt Cond,Ordinal,Evaluates the general condition of the basement
9,Bsmt Exposure,Ordinal,Refers to walkout or garden level walls


In [ ]:
joined.shape

## Columns in Data Dictionary but not in Data


In [25]:
(
cols_df
    .assign( foo = 1 )
    .set_index(0)
    .join( data_dictionary.set_index( "name" ), how="right" )
    .reset_index( names=["col"])
    .query( "foo.isnull()" )
    .drop(columns=["foo"])
    .reset_index( drop = True )
)

,col,type,description
0,Order,Discrete,Observation number
1,Exterior 1,Nominal,Exterior covering on house
2,Exterior 2,Nominal,Exterior covering on house
3,BsmtFinType 2,Ordinal,Rating of basement finished area
4,HeatingQC,Ordinal,Heating quality and condition
5,Bedroom,Discrete,Bedrooms above grade
6,Kitchen,Discrete,Kitchens above grade
7,KitchenQual,Ordinal,Kitchen quality
8,TotRmsAbvGrd,Discrete,Total rooms above grade
9,FireplaceQu,Ordinal,Fireplace quality


In [27]:
(
cols_df
    .assign( data = "y" )
    .set_index(0)
    .join( data_dictionary.set_index( "name" ), how="outer" )
    .query( "data.isnull() or type.isnull()" )
    .reset_index( names=["col"])
    .reset_index( drop = True )
)

,col,data,type,description
0,3-Ssn Porch,NaN,Continuous,Three season porch area in square feet
1,3Ssn Porch,y,NaN,NaN
2,Bedroom,NaN,Discrete,Bedrooms above grade
3,Bedroom AbvGr,y,NaN,NaN
4,BsmtFin Type 2,y,NaN,NaN
5,BsmtFinType 2,NaN,Ordinal,Rating of basement finished area
6,Exterior 1,NaN,Nominal,Exterior covering on house
7,Exterior 1st,y,NaN,NaN
8,Exterior 2,NaN,Nominal,Exterior covering on house
9,Exterior 2nd,y,NaN,NaN
